In [48]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from sklearn.utils.class_weight import compute_class_weight

In [42]:
# CONFIGURATION
PROCESSED_DATA_DIR = 'brain_tumor_data_preprocessed/'
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16 # Smaller dataset so small batches
EPOCHS = 100

In [43]:
# DATA LOADING / AUGMENTATION using TensorFlow
train_datagen = ImageDataGenerator(
    rescale=1./255,         # Normalize pixel values to 0-1
    rotation_range=20,      # To help prevent overfitting
    horizontal_flip=True,   # To help prevent overfitting
    width_shift_range=0.1,  # More augmentations since dataset is smaller
    height_shift_range=0.1, # |
    shear_range=0.1,        # |
    zoom_range=0.1,         # |
    fill_mode='nearest',
    validation_split=0.2    # 80/20 method for training/testing
)
validation_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)
train_generator = train_datagen.flow_from_directory(
    PROCESSED_DATA_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training'
)

validation_generator = validation_datagen.flow_from_directory(
    PROCESSED_DATA_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation'
)

Found 164 images belonging to 2 classes.
Found 40 images belonging to 2 classes.


In [44]:
# Attempt at handling imbalance by using class weights
class_labels = np.unique(train_generator.classes)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=class_labels,
    y=train_generator.classes
)
class_weights_dict = {label: weight for label, weight in zip(class_labels, class_weights)}
print(f"Class Weights calculated: {class_weights_dict}")

Class Weights calculated: {np.int32(0): np.float64(1.3666666666666667), np.int32(1): np.float64(0.7884615384615384)}


In [13]:
# Architecture References: 
# https://www.codecademy.com/article/understanding-convolutional-neural-network-cnn-architecture
# https://developers.google.com/machine-learning/practica/image-classification/convolutional-neural-networks

In [45]:
class NeuroScanCNN(tf.keras.Model):
    def __init__(self):
        super(NeuroScanCNN, self).__init__()
        # Starting with two layers per referenced architecture
        self.convolutional_1 = Conv2D(32, (3,3), activation='relu')
        self.pooling_1 = MaxPooling2D((2,2))

        self.convolutional_2 = Conv2D(32, (3,3), activation='relu')
        self.pooling_2 = MaxPooling2D((2,2))

        # Classification layer
        self.flatten = Flatten()
        self.dense = Dense(128, activation='relu')
        self.dropout = Dropout(0.2) # More overfitting protection
        
        # Setting sigmoid for 0/1 output Note: Cannot use "output" in __init__
        self.output_layer = Dense(1, activation='sigmoid') 

    def forward(self, x):
        conv_x = self.convolutional_1(x)
        pool_x = self.pooling_1(conv_x)

        conv_x = self.convolutional_2(pool_x)
        pool_x = self.pooling_2(conv_x)

        flat_x = self.flatten(pool_x)
        dense_x = self.dense(flat_x)
        dense_x = self.dropout(dense_x)
        return self.output(dense_x)  

In [46]:
# Create and compile model
model = NeuroScanCNN()
# Compiling with Binary Cross Entropy loss and accuracy as the main metric per README,
# added in precision and recall as well as dataset is imbalanced. Plus, high recall is important
# for a model like this.
model.compile(
    optimizer='adam',
    loss='binary_crossentrophy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
)

In [49]:
model.summary()

Model: "neuro_scan_cnn_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_24 (Conv2D)                   │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_24 (MaxPooling2D)      │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_25 (Conv2D)                   │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_25 (MaxPooling2D)      │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_12 (Flatten)                 │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_24 (Dense)                     │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_12 (Dropout)                 │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_25 (Dense)                     │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)